# 常见问题和陷阱

常规矢量拟合法容易受到一些普遍问题和特定用户错误的影响。本节旨在收集并解决常见问题，并帮助减轻这些问题。可以在[矢量拟合教程](../../tutorials/VectorFitting.ipynb)中找到更多说明和背景信息。

## 问题来源

以下因素可能会在矢量拟合过程中引起严重问题，并可能导致拟合收敛性差和/或拟合误差增大：1. 极点数量不足2. 极点数量过多3. 数据中存在强噪声从第 1) 和第 2) 点可以看出，必须仔细选择极点数量，这通常需要进行多次尝试才能获得正确的结果。但是，即使选择了正确的极点数量，如果数据中存在强噪声（通常来自测量），拟合质量和所需的迭代次数也可能会令人失望。在这种情况下，仍然可以使用 `auto_fit()` 来实现成功的矢量拟合，`auto_fit()` 是矢量拟合算法的改进和自动化版本，它利用迭代极点添加和筛选。示例 2 中的 2 端口测量是一个难以正确拟合的网络。它将作为不同问题的示例。此外，请参阅 [示例 2：测量的 190 GHz 有源 2 端口](vectorfitting_ex2_190ghz_active.ipynb)，了解在此网络上演示 `auto_fit()` 的用法。

In [ ]:
import matplotlib.pyplot as mplt
import numpy as np

import skrf
from skrf.vectorFitting import VectorFitting

nw = skrf.network.Network('./190ghz_tx_measured.S2P')
vf = VectorFitting(nw)
freqs = np.linspace(100e9, 300e9, 401)

## 噪声数据示例，极点数量不足

三个实极点和一个共轭复极点不足以对该网络进行准确拟合。它需要令人惊讶的多次迭代，并且最终模型的误差仍然相当大：

In [ ]:
vf.vector_fit(n_poles_real=3, n_poles_cmplx=1)

In [ ]:
print(f'RMS error = {vf.get_rms_error()}')

In [ ]:
vf.plot_convergence()

In [ ]:
vf.plot_s_db(freqs=freqs)

## 极点过多导致的数据噪声示例

对于这个简单的网络来说，十个实极点和十个共轭复极点太多了。未使用的极点会被移到更高的频率，超出测量频带，并且在重新定位过程中仍然会产生干扰。由于这些多余的极点，重新定位过程无法正确收敛，并在达到最大迭代次数（默认为 100）后停止。有趣的是，残差在最初的 20 次迭代中会相对快速地稳定下来，如图中的收敛图所示。尽管如此，在测量频带内的拟合结果非常好，但可以明显看到超出频带的多余极点。

In [ ]:
vf.vector_fit(n_poles_real=10, n_poles_cmplx=10)

In [ ]:
vf.plot_convergence()

In [ ]:
print(f'RMS error = {vf.get_rms_error()}')

In [ ]:
vf.plot_s_db(freqs=freqs)

## 关于起始极点的一些说明

在极点重新定位过程中（拟合过程的第一步），起始极点会依次移动到频率上，以便它们能够最好地匹配所有目标响应。此外，极点的类型可以从实数变为复共轭：两个实极点可以变为一个复共轭极点（反之亦然）。因此，存在多种起始极点的组合，可以产生相同的最终极点集合。但是，某些设置的收敛速度会比其他设置更快，这也取决于初始极点间距。在极端情况下，如果两个实极点表现得完全像一个复共轭极点，该算法甚至可能“无法确定”，并且会“卡住”，在没有收敛到最终解的情况下反复跳动。对于 `n_poles_real=3, n_poles_cmplx=4`（即 3+4）的一次拟合尝试，等效的设置如下：- 1+5- **3+4**- 5+3- 7+2- 9+1- 11+0对于另一次 `n_poles_real=4, n_poles_cmplx=4`（即 4+4）的拟合尝试，等效的设置如下：- 0+6- 2+5- **4+4**- 6+3- 8+2- 10+1- 12+0以下是一些由于两个（同样好的）解之间的振荡而无法正确收敛的问题设置示例：- 0+5 <--> 2+4 <--> ...- 0+7 <--> 2+5 <--> ...

In [ ]:
vf.vector_fit(n_poles_real=0, n_poles_cmplx=5)
vf.plot_convergence()

即使极点移动过程在两个（或更多？）解之间振荡，并且没有收敛，拟合仍然是成功的，因为这些解本身确实收敛了：

In [ ]:
vf.get_rms_error()

In [ ]:
fig, ax = mplt.subplots(2, 2)
fig.set_size_inches(12, 8)
vf.plot_s_mag(0, 0, freqs=freqs, ax=ax[0][0]) # s11
vf.plot_s_mag(0, 1, freqs=freqs, ax=ax[0][1]) # s12
vf.plot_s_mag(1, 0, freqs=freqs, ax=ax[1][0]) # s21
vf.plot_s_mag(1, 1, freqs=freqs, ax=ax[1][1]) # s22
fig.tight_layout()
mplt.show()